# Brain Tumor MRI Dataset — Phase 1 Exploration

This notebook performs a read-only audit of the original dataset. It does not preprocess images, create new splits, or train a model. All reported values are calculated directly from the files under `data/Training` and `data/Testing`.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, ImageOps, ExifTags
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'data' / 'Training').is_dir()
)
DATA_DIR = PROJECT_ROOT / 'data'
SPLITS = ('Training', 'Testing')
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATA_DIR}')

Project root: .
Dataset root: data


## Folder and class discovery

Classes are discovered from subdirectories rather than assumed in advance.

In [2]:
split_dirs = {split: DATA_DIR / split for split in SPLITS}
classes_by_split = {
    split: sorted(path.name for path in folder.iterdir() if path.is_dir())
    for split, folder in split_dirs.items()
}
display(pd.Series({k: ', '.join(v) for k, v in classes_by_split.items()}, name='classes').to_frame())
assert classes_by_split['Training'] == classes_by_split['Testing'], 'Split classes differ'
CLASSES = classes_by_split['Training']

,classes
Training,"glioma, meningioma, notumor, pituitary"
Testing,"glioma, meningioma, notumor, pituitary"


## Full image integrity and metadata scan

Every file is decoded with Pillow. Raw SHA-256 identifies byte-identical files. A second hash uses decoded RGB pixels plus dimensions, so it can find visually identical images stored with different container metadata or color modes.

In [3]:
records = []
corrupted = []

for split in SPLITS:
    for class_name in CLASSES:
        class_dir = DATA_DIR / split / class_name
        for path in sorted(p for p in class_dir.iterdir() if p.is_file()):
            row = {
                'split': split,
                'class': class_name,
                'filename': path.name,
                'path': str(path),
                'extension': path.suffix.lower(),
                'size_bytes': path.stat().st_size,
            }
            try:
                with Image.open(path) as image:
                    image.verify()
                with Image.open(path) as image:
                    image.load()
                    rgb = image.convert('RGB')
                    pixel_payload = f'{rgb.width}x{rgb.height}:'.encode() + rgb.tobytes()
                    row.update({
                        'width': image.width,
                        'height': image.height,
                        'aspect_ratio': image.width / image.height,
                        'format': image.format,
                        'mode': image.mode,
                        'raw_sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
                        'pixel_sha256': hashlib.sha256(pixel_payload).hexdigest(),
                        'exif_tags': tuple(sorted(ExifTags.TAGS.get(k, str(k)) for k in image.getexif())),
                    })
                records.append(row)
            except Exception as exc:
                corrupted.append({'path': str(path), 'error': repr(exc)})

images = pd.DataFrame(records)
corrupted_df = pd.DataFrame(corrupted)
print(f'Successfully decoded: {len(images):,}')
print(f'Corrupted/unreadable: {len(corrupted_df):,}')
display(corrupted_df if not corrupted_df.empty else pd.DataFrame({'result': ['No corrupted images found']}))

Successfully decoded: 7,200
Corrupted/unreadable: 0


,result
0,No corrupted images found


## Class distribution and imbalance

In [4]:
distribution = (
    images.groupby(['class', 'split']).size().unstack(fill_value=0)
    .reindex(CLASSES)
)
distribution['Total'] = distribution.sum(axis=1)
distribution.loc['Total'] = distribution.sum(axis=0)
display(distribution)

class_counts = images.groupby(['split', 'class']).size().rename('count').reset_index()
class_counts['share_pct'] = class_counts.groupby('split')['count'].transform(lambda s: 100 * s / s.sum())
display(class_counts)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=class_counts, x='class', y='count', hue='split', ax=ax)
ax.set(title='Images per class and split', xlabel='Class', ylabel='Image count')
plt.tight_layout()
plt.show()

split,Testing,Training,Total
class,,,
glioma,400,1400,1800
meningioma,400,1400,1800
notumor,400,1400,1800
pituitary,400,1400,1800
Total,1600,5600,7200


,split,class,count,share_pct
0,Testing,glioma,400,25.0
1,Testing,meningioma,400,25.0
2,Testing,notumor,400,25.0
3,Testing,pituitary,400,25.0
4,Training,glioma,1400,25.0
5,Training,meningioma,1400,25.0
6,Training,notumor,1400,25.0
7,Training,pituitary,1400,25.0


<Figure size 900x450 with 1 Axes>

## Image dimensions, formats, modes, and file sizes

In [5]:
dimension_summary = pd.DataFrame({
    'metric': ['unique width-height pairs', 'min width', 'median width', 'max width',
               'min height', 'median height', 'max height', 'square images', 'non-square images'],
    'value': [
        images[['width', 'height']].drop_duplicates().shape[0],
        images['width'].min(), images['width'].median(), images['width'].max(),
        images['height'].min(), images['height'].median(), images['height'].max(),
        (images['width'] == images['height']).sum(),
        (images['width'] != images['height']).sum(),
    ],
})
display(dimension_summary)

top_dimensions = images.groupby(['width', 'height']).size().sort_values(ascending=False).head(15).rename('count').reset_index()
display(top_dimensions)
display(images.groupby(['extension', 'format']).size().rename('count').to_frame())
display(images['mode'].value_counts().rename_axis('mode').to_frame('count'))
display(images['size_bytes'].describe().to_frame().T)

mislabeled = images.loc[images['extension'].eq('.jpg') & images['format'].ne('JPEG'),
                         ['split', 'class', 'filename', 'extension', 'format', 'mode', 'width', 'height']]
print(f'Files whose .jpg suffix disagrees with decoded format: {len(mislabeled)}')
display(mislabeled)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(images['width'], bins=35, ax=axes[0], color='#4C72B0')
axes[0].set_title('Width distribution')
sns.histplot(images['height'], bins=35, ax=axes[1], color='#DD8452')
axes[1].set_title('Height distribution')
plt.tight_layout()
plt.show()

,metric,value
0,unique width-height pairs,447.0
1,min width,150.0
2,median width,512.0
3,max width,1375.0
4,min height,167.0
5,median height,512.0
6,max height,1446.0
7,square images,5667.0
8,non-square images,1533.0


,width,height,count
0,512,512,5014
1,225,225,338
2,630,630,90
3,201,251,57
4,228,221,51
5,232,217,50
6,442,442,48
7,236,236,48
8,150,198,44
9,200,252,43


count
extension format       
.jpg      JPEG     7196
          PNG         4

,count
mode,
RGB,4129
L,3067
RGBA,3
P,1


,count,mean,std,min,25%,50%,75%,max
size_bytes,7200.0,23312.355139,16300.958173,3681.0,16113.5,21705.5,27392.25,306492.0


Files whose .jpg suffix disagrees with decoded format: 4


,split,class,filename,extension,format,mode,width,height
3025,Training,notumor,Tr-no_1200.jpg,.jpg,PNG,RGBA,393,400
3335,Training,notumor,Tr-no_22.jpg,.jpg,PNG,RGBA,550,664
3726,Training,notumor,Tr-no_572.jpg,.jpg,PNG,RGBA,442,454
3785,Training,notumor,Tr-no_625.jpg,.jpg,PNG,P,728,725


<Figure size 1200x450 with 2 Axes>

## Sample MRI images

A fixed random seed makes the displayed Training examples reproducible. Images are shown in their original form; no transformations are saved.

In [6]:
rng = np.random.default_rng(42)
samples_per_class = 3
fig, axes = plt.subplots(len(CLASSES), samples_per_class, figsize=(10, 11))
for row_index, class_name in enumerate(CLASSES):
    candidates = images[(images['split'] == 'Training') & (images['class'] == class_name)]
    selected = candidates.iloc[rng.choice(len(candidates), size=samples_per_class, replace=False)]
    for col_index, (_, sample) in enumerate(selected.iterrows()):
        with Image.open(sample['path']) as image:
            axes[row_index, col_index].imshow(image, cmap='gray')
        axes[row_index, col_index].set_title(f"{class_name}\n{sample['filename']}", fontsize=9)
        axes[row_index, col_index].axis('off')
plt.suptitle('Reproducible sample of Training images', y=1.01)
plt.tight_layout()
plt.show()

<Figure size 1000x1100 with 12 Axes>

## Duplicate and leakage audit

Raw hashes detect identical files. Decoded-pixel hashes detect exact RGB pixel equality while preserving original dimensions. The second check is intentionally encoding-independent and is the more relevant leakage check here.

In [7]:
def duplicate_groups(frame, hash_column):
    return [group.copy() for _, group in frame.groupby(hash_column) if len(group) > 1]

def cross_split_pairs(frame, hash_column):
    pairs = []
    for _, group in frame.groupby(hash_column):
        training = group[group['split'] == 'Training']
        testing = group[group['split'] == 'Testing']
        for _, train_row in training.iterrows():
            for _, test_row in testing.iterrows():
                pairs.append({
                    'train_class': train_row['class'],
                    'train_file': train_row['filename'],
                    'test_class': test_row['class'],
                    'test_file': test_row['filename'],
                    'label_conflict': train_row['class'] != test_row['class'],
                    'hash': train_row[hash_column],
                })
    return pd.DataFrame(pairs)

raw_groups = duplicate_groups(images, 'raw_sha256')
raw_cross = cross_split_pairs(images, 'raw_sha256')
pixel_cross = cross_split_pairs(images, 'pixel_sha256')

raw_duplicate_files = sum(len(group) for group in raw_groups)
raw_redundant_copies = sum(len(group) - 1 for group in raw_groups)
pixel_cross_groups = pixel_cross['hash'].nunique() if not pixel_cross.empty else 0

duplicate_summary = pd.DataFrame({
    'check': [
        'Raw duplicate groups anywhere', 'Files in raw duplicate groups',
        'Redundant raw copies', 'Raw cross-split pairs',
        'Decoded-pixel cross-split groups', 'Decoded-pixel cross-split pairs',
        'Distinct Training files in pixel pairs', 'Distinct Testing files in pixel pairs',
        'Cross-label pixel pairs',
    ],
    'value': [
        len(raw_groups), raw_duplicate_files, raw_redundant_copies, len(raw_cross),
        pixel_cross_groups, len(pixel_cross),
        pixel_cross['train_file'].nunique() if not pixel_cross.empty else 0,
        pixel_cross['test_file'].nunique() if not pixel_cross.empty else 0,
        int(pixel_cross['label_conflict'].sum()) if not pixel_cross.empty else 0,
    ],
})
display(duplicate_summary)

if not pixel_cross.empty:
    display(pixel_cross.groupby(['train_class', 'test_class']).size().rename('pairs').to_frame())
    display(pixel_cross.drop(columns='hash').head(20))

,check,value
0,Raw duplicate groups anywhere,153
1,Files in raw duplicate groups,340
2,Redundant raw copies,187
3,Raw cross-split pairs,0
4,Decoded-pixel cross-split groups,111
5,Decoded-pixel cross-split pairs,121
6,Distinct Training files in pixel pairs,115
7,Distinct Testing files in pixel pairs,114
8,Cross-label pixel pairs,0


,,pairs
train_class,test_class,
glioma,glioma,5
meningioma,meningioma,107
notumor,notumor,9


,train_class,train_file,test_class,test_file,label_conflict
0,meningioma,Tr-me_36.jpg,meningioma,Te-me_1.jpg,False
1,meningioma,Tr-me_479.jpg,meningioma,Te-me_278.jpg,False
2,meningioma,Tr-me_404.jpg,meningioma,Te-me_101.jpg,False
3,notumor,Tr-no_760.jpg,notumor,Te-no_315.jpg,False
4,meningioma,Tr-me_1290.jpg,meningioma,Te-me_37.jpg,False
5,meningioma,Tr-me_776.jpg,meningioma,Te-me_253.jpg,False
6,meningioma,Tr-me_331.jpg,meningioma,Te-me_18.jpg,False
7,meningioma,Tr-me_1057.jpg,meningioma,Te-me_270.jpg,False
8,meningioma,Tr-me_145.jpg,meningioma,Te-me_296.jpg,False
9,notumor,Tr-no_970.jpg,notumor,Te-no_159.jpg,False


## Patient IDs, metadata, filenames, and provenance signals

In [8]:
non_image_files = [
    str(path.relative_to(PROJECT_ROOT))
    for path in PROJECT_ROOT.rglob('*')
    if path.is_file() and path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}
    and '.ipynb_checkpoints' not in path.parts
]
filename_patterns = images['filename'].map(lambda name: re.sub(r'\d+', '{n}', name)).value_counts()
augmented_names = images[images['filename'].str.contains('aug', case=False)].groupby(['split', 'class']).size()
exif_images = images[images['exif_tags'].map(bool)]
exif_tag_counts = Counter(tag for tags in exif_images['exif_tags'] for tag in tags)

print('Filename patterns:')
display(filename_patterns.rename('count').to_frame())
print('Files explicitly marked as augmented:')
display(augmented_names.rename('count').to_frame())
print(f'Images containing EXIF metadata: {len(exif_images)}')
display(pd.Series(exif_tag_counts).sort_values(ascending=False).rename('image_count').to_frame())
print('No filename is shared by Training and Testing:',
      not bool(set(images.loc[images.split == 'Training', 'filename']) &
               set(images.loc[images.split == 'Testing', 'filename'])))
print('No patient/study identifier is encoded by the observed split-class-sequence filename scheme.')
print('Non-image project files (no supplied patient/scan metadata table was found):')
display(pd.DataFrame({'path': sorted(non_image_files)}))

Filename patterns:


,count
filename,
Tr-gl_{n}.jpg,1400
Tr-no_{n}.jpg,1400
Tr-pi_{n}.jpg,1400
Tr-me_{n}.jpg,1300
Te-gl_{n}.jpg,400
Te-no_{n}.jpg,400
Te-pi_{n}.jpg,400
Te-me_{n}.jpg,297
Te-aug-me_{n}.jpg,103


Files explicitly marked as augmented:


,,count
split,class,
Testing,meningioma,103
Training,meningioma,100


Images containing EXIF metadata: 22


,image_count
ExifOffset,21
Orientation,20
Software,19
59932,7
DateTime,6
ProcessingSoftware,5
ResolutionUnit,2
XResolution,2
YResolution,2
Artist,2


No filename is shared by Training and Testing: True
No patient/study identifier is encoded by the observed split-class-sequence filename scheme.
Non-image project files (no supplied patient/scan metadata table was found):


,path
0,.DS_Store
1,.gitignore
2,README.md
3,models/.gitkeep
4,notebooks/01_data_exploration.ipynb
5,reports/figures/.gitkeep
6,requirements.txt
7,src/.gitkeep


## Phase 1 conclusion

The classes are exactly balanced and every image decodes, but this dataset is not ready for a defensible model benchmark. The decoded-pixel audit confirms visually exact images across Training and Testing. Patient-level separation cannot be verified because patient/study IDs are absent. Files explicitly marked as augmented occur in Testing, and repeated images plus heterogeneous dimensions, color modes, and encodings create additional shortcut-learning and provenance risks.

Stop here. Deduplication, split reconstruction, preprocessing, augmentation design, and model training belong to later phases and have not been performed.